# Gaussian Processes Summer School

## Part 6: Non-stationary Spectral GPLVMs

<a href="https://www.uc3m.es/"><img src="./uc3m.png" alt="Universidad Carlos III de Madrid" width="520"></a>

**Author:** [Pablo M. Olmos](https://olmos-pm.github.io/)

**Email:** [pamartin@ing.uc3m.es](mailto:pamartin@ing.uc3m.es)

---

Part 4 introduced an amortized variational GPLVM with three stationary spectral families. We now give each family a matched non-stationary counterpart using the paired-frequency construction from Part 5. Every image $\mathbf y_n\in\mathbb R^D$ receives a learned coordinate $\mathbf x_n\in\mathbb R^Q$, and labels remain completely absent during representation learning.

### Learning goals

1. Extend the stationary RFF-GPLVM from Part 4 with a non-stationary joint spectrum $p_\theta(\mathbf w_1,\mathbf w_2)$.
2. Construct differentiable paired-frequency samples for Gaussian, Gaussian-mixture, and implicit neural spectra.
3. Compare each stationary model with its non-stationary counterpart under the same encoder, data, and frequency budget.
4. Evaluate latent geometry using t-SNE and held-out $k$-NN classification accuracy.



## 1. GPLVM and variational inference


Let $Y=[\mathbf y_1^\top,\ldots,\mathbf y_N^\top]^\top\in\mathbb R^{N\times D}$, where e.g. $D=784$ for MNIST, and let $X\in\mathbb R^{N\times Q}$ with $Q=10$ a low dimensional matrix of **latent codes or vectors** that define each of the $N$ datapoints in the latent space $\mathcal{X}$. A GPLVM reverses the usual regression roles:

$$\mathbf x_n\sim\mathcal N(0,I),\qquad
f_d(\cdot)\sim\mathcal{GP}(0,k_\theta),\qquad
y_{nd}=f_d(\mathbf x_n)+\epsilon_{nd}.$$

The $D$ pixel functions are conditionally independent but share the same latent inputs and
kernel. Consequently, $p(Y\mid X)=\prod_{d=1}^D\mathcal N(\mathbf y_{:d};0,K_X+\sigma^2I)$. For a stationary kernel, Bochner's theorem gives

$$k_\theta(\mathbf x,\mathbf x')=\mathbb E_{\mathbf w\sim p_\theta}[\cos(\mathbf w^\top(\mathbf x-\mathbf x'))].$$

After drawing $L$ frequencies $\mathbf w_l\sim p_\theta$, we use the paired feature map

$$\phi_\theta(\mathbf x)=\frac1{\sqrt L}[\cos(\mathbf w_1^\top\mathbf x),\sin(\mathbf w_1^\top\mathbf x),\ldots,\cos(\mathbf w_L^\top\mathbf x),\sin(\mathbf w_L^\top\mathbf x)]^\top.$$

There are $2L$ real features. This construction is stationary because every feature inner product depends on $\mathbf x-\mathbf x'$.

For a non-stationary kernel, Part 5 replaced the single spectral variable by a pair $(\mathbf w_1,\mathbf w_2)$ drawn from a bivariate measure. Many spectral constructions for non-stationary kernels exist; here we use the one from our NeurIPS 2025 paper because its real random-feature map is especially convenient. We focus on the case where the bivariate measure admits a joint density $p_\theta(\mathbf w_1,\mathbf w_2)$, although this need not hold in general. Drawing $L$ pairs gives

$$
\phi_\theta(\mathbf x)=\sqrt{\frac{1}{4L}}
\begin{bmatrix}
\cos(\mathbf w_{1}^{(1)\top}\mathbf x)+\cos(\mathbf w_{2}^{(1)\top}\mathbf x)\\
\vdots\\
\cos(\mathbf w_{1}^{(L)\top}\mathbf x)+\cos(\mathbf w_{2}^{(L)\top}\mathbf x)\\[2pt]
\sin(\mathbf w_{1}^{(1)\top}\mathbf x)+\sin(\mathbf w_{2}^{(1)\top}\mathbf x)\\
\vdots\\
\sin(\mathbf w_{1}^{(L)\top}\mathbf x)+\sin(\mathbf w_{2}^{(L)\top}\mathbf x)
\end{bmatrix}.
$$

When $\mathbf w_1\neq\mathbf w_2$, the inner product can depend separately on $\mathbf x$ and $\mathbf x'$, rather than only on their difference. It is nevertheless positive semidefinite because it is explicitly an inner product. Both stationary and non-stationary versions still have $2L$ real features, so the matrix determinant lemma and Woodbury identity reduce the likelihood calculation to a $2L\times2L$ system; the implementation never constructs an $N\times N$ covariance matrix.

### Learn the inverse mapping (from $\mathbf y$ to $\mathbf x$) with variational inference
Unlike the original GPLVM, we do not store one free variational mean and variance per training
example. A shared encoder reads an **observed datum** and outputs

$$q_\phi(\mathbf x_n\mid\mathbf y_n)
=\mathcal N\!\left(\boldsymbol\mu_\phi(\mathbf y_n),
\operatorname{diag}(\boldsymbol\sigma_\phi^2(\mathbf y_n))\right).$$

where $\boldsymbol \mu_\phi$ and $\boldsymbol\sigma_\phi^2$ are the outputs of a Neural Network with paramter set $\phi$ and input $\mathbf y$. This is amortized inference: the same network can embed a previously unseen image. Notice that the network input is $\mathbf y_n$, not the unknown $\mathbf x_n$.

#### The Evidence Lower Bound (ELBO)

We choose $q(W)=p_\theta(W)$ in stationary models and $q(W_1,W_2)=p_\theta(W_1,W_2)$ in paired models, drawing fresh reparameterized frequencies during training. Thus
$\mathrm{KL}[q(W)\|p(W)]=0$. The resulting ELBO balances how well we probabilistically reconstruct our observations from posterior inference, while penalizes solutions far away from our prior $p(\mathbf x)$:

$$\mathcal L=
\underbrace{\mathbb E_{q_\phi(X\mid Y)p_\theta(W)}
[\log p(Y\mid X,W)]}_{\text{reconstruction term}}
-\mathrm{KL}[q_\phi(X\mid Y)\|p(X)].$$

We estimate the expectation with one reparameterized draw
$X=\mu_\phi(Y)+\sigma_\phi(Y)\odot\varepsilon$ and one draw $W\sim p_\theta(W)$ per optimization
step. Learning $\theta$ is empirical Bayes: gradients pass through samples from the corresponding spectral distribution.
Tying $q(W)$ to the prior is simple and cheap, but it does **not** learn a data-adapted posterior
over individual frequencies.


## 2. Generative-model plate diagram

The plate diagram below summarizes the generative direction. White nodes are latent variables or functions, the gray node is observed, and the overlapping plates indicate repetition over observations $n=1,\ldots,N$ and output dimensions $d=1,\ldots,D$.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7.5)
ax.set_aspect("equal")
ax.axis("off")

# The overlapping plates encode the Cartesian indexing of y_nd.
n_plate = Rectangle((2.0, 0.8), 6.8, 3.7, fill=False, linewidth=1.8, edgecolor="C0")
d_plate = Rectangle((4.0, 2.0), 4.8, 3.8, fill=False, linewidth=1.8, edgecolor="C2")
ax.add_patch(n_plate)
ax.add_patch(d_plate)
ax.text(8.45, 1.02, r"$n=1,\ldots,N$", color="C0", ha="right", fontsize=12)
ax.text(8.45, 5.48, r"$d=1,\ldots,D$", color="C2", ha="right", fontsize=12)

positions = {
    "theta": (1.2, 6.4),
    "W": (4.2, 6.4),
    "noise": (0.8, 3.0),
    "x": (3.2, 1.9),
    "beta": (6.5, 5.1),
    "y": (6.8, 3.0),
}
labels_nodes = {
    "theta": r"$\theta$",
    "W": r"$W$",
    "noise": r"$\sigma^2$",
    "x": r"$\mathbf{x}_n$",
    "beta": r"$\boldsymbol{\beta}_d$",
    "y": r"$y_{nd}$",
}
for name, position in positions.items():
    observed = name == "y"
    node = Circle(
        position, 0.48, facecolor="0.78" if observed else "white",
        edgecolor="black", linewidth=1.8, zorder=3,
    )
    ax.add_patch(node)
    ax.text(*position, labels_nodes[name], ha="center", va="center", fontsize=14, zorder=4)

def arrow(source, target):
    ax.add_patch(FancyArrowPatch(
        positions[source], positions[target], arrowstyle="-|>",
        mutation_scale=16, linewidth=1.5, color="black",
        shrinkA=25, shrinkB=25, zorder=2,
    ))

arrow("theta", "W")
arrow("W", "y")
arrow("beta", "y")
arrow("x", "y")
arrow("noise", "y")

ax.text(5.0, 7.15, "Spectral RFF-GPLVM generative model", ha="center", fontsize=15, weight="bold")
ax.text(5.75, 5.92, r"$W=\{(\mathbf{w}_{1l},\mathbf{w}_{2l})\}_{l=1}^L\sim p_\theta$", ha="center", fontsize=10)
ax.text(3.2, 1.25, r"$\mathbf{x}_n\sim\mathcal{N}(0,I_K)$", ha="center", fontsize=10)
ax.text(7.5, 4.58, r"$\boldsymbol{\beta}_d\sim\mathcal{N}(0,I_{2L})$", ha="center", fontsize=10)
ax.text(7.5, 2.25, r"$y_{nd}=\sqrt{\sigma_f^2}\,\phi_W(\mathbf{x}_n)^\top\boldsymbol{\beta}_d+\epsilon_{nd}$", ha="center", fontsize=9)
fig.tight_layout()
plt.show()


This is the **Bayesian linear regression representation of the RFF approximation**. In a stationary model, $W$ contains single frequencies $\mathbf w_l$; in a non-stationary model, it contains pairs $(\mathbf w_{1l},\mathbf w_{2l})$. These global spectral variables are drawn from the learned prior and shared by all $D$ output functions. Given $W$, each latent coordinate $\mathbf x_n$ determines the same feature vector $\phi_W(\mathbf x_n)\in\mathbb R^{2L}$ for every output dimension. In contrast, each output $d$ has its own random coefficient vector

$$\boldsymbol{\beta}_d\sim\mathcal N(0,I_{2L}),\qquad y_{nd}=\sqrt{\sigma_f^2}\,\phi_W(\mathbf x_n)^\top\boldsymbol{\beta}_d+\epsilon_{nd}.$$

Therefore, $W$ is outside both plates, $\phi_W(\mathbf x_n)$ is indexed only by $n$, $\boldsymbol{\beta}_d$ lies only inside the $D$-plate, and $y_{nd}$ lies in the overlap of the $N$- and $D$-plates. The implementation never explicitly samples the coefficient vectors: integrating out $\boldsymbol{\beta}_d$ recovers the approximate GP covariance

$$K_X\approx\sigma_f^2\Phi_W(X)\Phi_W(X)^\top.$$

Thus, all pixel functions are independent conditional on the shared feature matrix but have the same covariance. The encoder $q_\phi(\mathbf x_n\mid\mathbf y_n)$ is an inference mechanism and therefore is not part of this generative-model diagram.


## 3. Matched stationary and non-stationary spectra

### Gaussian family

$$p_\theta(\mathbf w)=\mathcal N(\mathbf0,\operatorname{diag}(\boldsymbol\ell^{-2})),$$

The stationary model uses this Gaussian density, which implies the ARD RBF kernel

$$k_{\mathrm{RBF}}(\boldsymbol\tau)=\sigma_f^2\exp\left[-\frac12\sum_{j=1}^K\frac{\tau_j^2}{\ell_j^2}\right].$$

Its non-stationary counterpart learns one joint Gaussian over $(\mathbf w_1,\mathbf w_2)$. For each latent dimension $j$,

$$
w_{1j}=\mu_{1j}+s_{1j}\epsilon_{1j},\qquad
w_{2j}=\mu_{2j}+s_{2j}\left(\rho_j\epsilon_{1j}+\sqrt{1-\rho_j^2}\epsilon_{2j}\right).
$$

This pathwise parameterization gives differentiable correlated samples while enforcing $s_{ij}>0$ and $|\rho_j|<1$.

### Gaussian-mixture family

The second model learns

$$k_{\mathrm{mix}}(\boldsymbol\tau)=\sigma_f^2\sum_{m=1}^M\rho_m\exp\left[-\frac12\sum_{j=1}^K s_{mj}^2\tau_j^2\right]\cos(\boldsymbol\mu_m^\top\boldsymbol\tau),$$

with $\rho_m=\operatorname{softmax}(\alpha)_m$. We draw frequencies separately from every Gaussian component and concatenate feature blocks scaled by $\sqrt{\rho_m/L_m}$. Thus the feature inner product contains each $\rho_m$ exactly once. The categorical component is summed analytically, so no Gumbel--Softmax or discrete component sample is used.

The non-stationary counterpart replaces each Gaussian over $\mathbf w$ with a correlated joint Gaussian over $(\mathbf w_1,\mathbf w_2)$ and uses the Part 5 feature map inside every component. Mixture weights are still marginalized exactly by weighted feature blocks.

### Implicit neural family

For each frequency, draw $\mathbf z_l\sim\mathcal N(\mathbf0,I_R)$ and transform it as

$$\mathbf w_l=a_\theta g_\theta(\mathbf z_l),$$

where $g_\theta$ is an MLP with SiLU activations and $a_\theta>0$ is learned. Paired sine/cosine features implicitly symmetrize the emitted spectrum because $+\mathbf w$ and $-\mathbf w$ produce the same stationary kernel contribution.

For the non-stationary counterpart, the network emits $2K$ values and is split into $(\mathbf w_1,\mathbf w_2)$. It therefore learns an implicit joint spectral distribution without evaluating a density. In all three comparisons, the stationary and non-stationary members receive the same total frequency budget $L$ and produce the same number $2L$ of real features.

A GPLVM can partly absorb geometry into its learned latent coordinates, so kernel stationarity is less directly identifiable here than in Part 5, where inputs were observed. The experiment asks an empirical question: under an otherwise matched training and evaluation protocol, does the additional joint-spectral flexibility yield latent neighborhoods that generalize better?


In [ ]:
import gzip
import random
import struct 
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    silhouette_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from train_amortized_gplvm_v2 import (
    ImplicitNeuralSpectralPrior,
    PairedLocallyPeriodicMixturePrior,
    PairedRBFSpectralPrior,
    PairedRFFGPLVM,
    decode_posterior_mean,
    decode_posterior_sample,
    fit_rff_posterior,
    latent_means,
    train_model,
)
from train_amortized_gplvm_part6 import (
    CorrelatedGaussianJointSpectralPrior,
    CorrelatedGaussianMixtureJointSpectralPrior,
    ImplicitJointSpectralPrior,
)

SEED = 52
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)


In [ ]:
# Main tunable settings
LATENT_DIM = 5                 # K
NUM_MIXTURE_COMPONENTS = 5     # M; shared by both mixture models
NUM_FOURIER_FREQUENCIES = 120  # L; every model produces 2L real features
IMPLICIT_NOISE_DIM = 5          # R (number of dimensions in the noise vector for the implicit neural spectral prior)
IMPLICIT_HIDDEN_DIMS = (16, 16, 16, 16)
NUM_EPOCHS = 3000 
PRINT_EVERY = 500
TRAIN_PER_CLASS = 1000 # Number of training samples per class
TEST_PER_CLASS = 50
TSNE_MAX_POINTS = 500
KNN_CANDIDATES = [1, 3, 5, 7, 9, 11, 15]
USE_FIXED_KNN = True            # True: always use FIXED_KNN_K; False: tune over KNN_CANDIDATES
FIXED_KNN_K = 1
CLUSTERING_SEEDS = (101, 102, 103, 104, 105)

print(
    f"K={LATENT_DIM}, M={NUM_MIXTURE_COMPONENTS}, "
    f"frequencies={NUM_FOURIER_FREQUENCIES}, "
    f"paired features={2 * NUM_FOURIER_FREQUENCIES}"
)


## 4. Data and evaluation protocol

The GPLVM is trained without labels on a balanced subset of the official MNIST training split. The encoder then embeds a disjoint balanced subset from the official test split. Labels are used only after GPLVM training:

- t-SNE visualizes the learned $K$-dimensional posterior means; it is not used for classification.
- `USE_FIXED_KNN=True` fixes the downstream classifier to $k=1$ (the active setting below). If the flag is disabled, `GridSearchCV` selects $k$ from `KNN_CANDIDATES` using 5-fold cross-validation. In either case, the classifier is evaluated once on the untouched test posterior means.
- To quantify the separation suggested by t-SNE, we standardize the original $K$-dimensional coordinates, fit ten-cluster $k$-means models on training embeddings, and evaluate their test assignments using adjusted mutual information (AMI) and adjusted Rand index (ARI). We average over the seeds in `CLUSTERING_SEEDS`.
- The class-label silhouette score measures test-class compactness and separation directly in the standardized latent space. It does not use t-SNE coordinates.
- We compare three matched families: Gaussian/RBF, Gaussian mixture, and implicit neural. Each family has a stationary spectrum $p(\mathbf w)$ and a non-stationary joint spectrum $p(\mathbf w_1,\mathbf w_2)$.
- All six models use the same two-hidden-layer MLP amortized encoder and receive identical images, latent dimension, frequency budget, optimization settings, and initial encoder weights. Labels are used only for downstream evaluation and plot coloring.


In [ ]:
DATASETS = {
    "MNIST": {
        "class_names": [str(i) for i in range(10)],
        "base_url": "https://storage.googleapis.com/cvdf-datasets/mnist/",
    }
}


def _balanced_indices(labels, per_class, rng):
    indices = np.concatenate([
        rng.choice(np.flatnonzero(labels == class_index), per_class, replace=False)
        for class_index in range(10)
    ])
    rng.shuffle(indices)
    return indices


def _read_idx_dataset(dataset_name, train):
    config = DATASETS[dataset_name]
    prefix = "train" if train else "t10k"
    filenames = {
        "images": f"{prefix}-images-idx3-ubyte.gz",
        "labels": f"{prefix}-labels-idx1-ubyte.gz",
    }
    raw_dir = Path("data") / dataset_name.replace("-", "") / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)

    def local_idx_path(filename):
        compressed = raw_dir / filename
        uncompressed = raw_dir / filename.removesuffix(".gz")
        if compressed.exists():
            return compressed, gzip.open
        if uncompressed.exists():
            return uncompressed, open
        print("downloading", dataset_name, filename)
        urllib.request.urlretrieve(config["base_url"] + filename, compressed)
        return compressed, gzip.open

    images_path, images_open = local_idx_path(filenames["images"])
    labels_path, labels_open = local_idx_path(filenames["labels"])
    with images_open(images_path, "rb") as stream:
        _, count, rows, cols = struct.unpack(">IIII", stream.read(16))
        pixels = np.frombuffer(stream.read(), dtype=np.uint8).reshape(count, rows * cols)
    with labels_open(labels_path, "rb") as stream:
        _, count = struct.unpack(">II", stream.read(8))
        labels = np.frombuffer(stream.read(), dtype=np.uint8, count=count).astype(np.int64)
    return torch.as_tensor(pixels.copy(), dtype=torch.float32) / 255.0, labels


def load_balanced_dataset(dataset_name, seed):
    if dataset_name != "MNIST":
        raise ValueError("Part 6 is configured for MNIST only")
    try:
        from torchvision.datasets import MNIST
        train_dataset = MNIST(root="data", train=True, download=True)
        test_dataset = MNIST(root="data", train=False, download=True)
        train_all = train_dataset.data.reshape(-1, 784).to(torch.float32) / 255.0
        test_all = test_dataset.data.reshape(-1, 784).to(torch.float32) / 255.0
        train_labels_all = np.asarray(train_dataset.targets)
        test_labels_all = np.asarray(test_dataset.targets)
    except (ImportError, RuntimeError):
        train_all, train_labels_all = _read_idx_dataset(dataset_name, train=True)
        test_all, test_labels_all = _read_idx_dataset(dataset_name, train=False)
    rng = np.random.default_rng(seed)
    train_indices = _balanced_indices(train_labels_all, TRAIN_PER_CLASS, rng)
    test_indices = _balanced_indices(test_labels_all, TEST_PER_CLASS, rng)
    train_raw = train_all[train_indices]
    test_raw = test_all[test_indices]
    pixel_mean = train_raw.mean(0, keepdim=True)
    return {
        "train_images": (train_raw - pixel_mean).to(device),
        "test_images": (test_raw - pixel_mean).to(device),
        "train_raw": train_raw,
        "test_raw": test_raw,
        "train_labels": train_labels_all[train_indices],
        "test_labels": test_labels_all[test_indices],
        "pixel_mean": pixel_mean.to(device),
        "class_names": DATASETS[dataset_name]["class_names"],
    }


In [ ]:
MODEL_SPECS = [
    {"name": "Gaussian | stationary RBF", "family": "Gaussian", "stationarity": "stationary"},
    {"name": "Gaussian | paired non-stationary", "family": "Gaussian", "stationarity": "non-stationary"},
    {"name": f"{NUM_MIXTURE_COMPONENTS}-Gaussian mixture | stationary", "family": "Gaussian mixture", "stationarity": "stationary"},
    {"name": f"{NUM_MIXTURE_COMPONENTS}-Gaussian mixture | paired non-stationary", "family": "Gaussian mixture", "stationarity": "non-stationary"},
    {"name": "Implicit neural | stationary", "family": "Implicit neural", "stationarity": "stationary"},
    {"name": "Implicit neural | paired non-stationary", "family": "Implicit neural", "stationarity": "non-stationary"},
]
MODEL_NAMES = [spec["name"] for spec in MODEL_SPECS]
MODEL_METADATA = {spec["name"]: spec for spec in MODEL_SPECS}


def tune_knn(z_train, y_train, z_test, y_test):
    candidate_neighbors = [FIXED_KNN_K] if USE_FIXED_KNN else KNN_CANDIDATES
    search = GridSearchCV(
        estimator=KNeighborsClassifier(),
        param_grid={"n_neighbors": candidate_neighbors},
        scoring="accuracy",
        cv=5,
        n_jobs=-1,
        refit=True,
    )
    search.fit(z_train, y_train)
    return {
        "best_k": int(search.best_params_["n_neighbors"]),
        "cv_accuracy": float(search.best_score_),
        "test_accuracy": float(search.score(z_test, y_test)),
    }


def latent_clustering_metrics(
    z_train, z_test, test_labels, *, num_clusters=10, seeds=CLUSTERING_SEEDS
):
    """Evaluate class separation in the original latent space, never in t-SNE."""
    scaler = StandardScaler()
    z_train_scaled = scaler.fit_transform(z_train)
    z_test_scaled = scaler.transform(z_test)

    ami_values, ari_values = [], []
    for clustering_seed in seeds:
        clustering = KMeans(
            n_clusters=num_clusters, n_init=10, random_state=clustering_seed
        )
        clustering.fit(z_train_scaled)  # labels are not used to fit the clusters
        test_clusters = clustering.predict(z_test_scaled)
        ami_values.append(
            adjusted_mutual_info_score(test_labels, test_clusters)
        )
        ari_values.append(adjusted_rand_score(test_labels, test_clusters))

    # This label-aware diagnostic measures class compactness/separation directly.
    class_silhouette = silhouette_score(z_test_scaled, test_labels)
    return {
        "AMI mean": float(np.mean(ami_values)),
        "AMI std": float(np.std(ami_values, ddof=1 if len(ami_values) > 1 else 0)),
        "ARI mean": float(np.mean(ari_values)),
        "ARI std": float(np.std(ari_values, ddof=1 if len(ari_values) > 1 else 0)),
        "class silhouette": float(class_silhouette),
    }


def tsne_projection(z, perplexity=30.0, steps=750, seed=SEED):
    # Exact small-data t-SNE implemented with NumPy and PyTorch only.
    z = np.asarray(z, dtype=np.float32)
    z = (z - z.mean(0)) / (z.std(0) + 1e-6)
    row_norms = np.sum(z * z, axis=1, keepdims=True)
    squared = np.maximum(row_norms + row_norms.T - 2.0 * z @ z.T, 0.0)
    n = len(z)
    conditional = np.zeros((n, n), dtype=np.float64)
    target_entropy = np.log(perplexity)
    for i in range(n):
        mask = np.arange(n) != i
        distances = squared[i, mask]
        low_log_precision, high_log_precision = -20.0, 20.0
        for _ in range(50):
            log_precision = (low_log_precision + high_log_precision) / 2.0
            probabilities = np.exp(-distances * np.exp(log_precision))
            probabilities /= probabilities.sum() + 1e-12
            entropy = -np.sum(probabilities * np.log(probabilities + 1e-12))
            if entropy > target_entropy:
                low_log_precision = log_precision
            else:
                high_log_precision = log_precision
        conditional[i, mask] = probabilities
    joint = (conditional + conditional.T) / (2.0 * n)
    joint = torch.as_tensor(joint, dtype=torch.float32, device=device).clamp_min(1e-12)

    generator = torch.Generator(device=device).manual_seed(seed)
    projection = torch.nn.Parameter(
        1e-4 * torch.randn(n, 2, generator=generator, device=device)
    )
    optimizer = torch.optim.Adam([projection], lr=0.5)
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        numerator = 1.0 / (1.0 + torch.cdist(projection, projection).square())
        off_diagonal = 1.0 - torch.eye(n, dtype=projection.dtype, device=device)
        numerator = numerator * off_diagonal
        q = (numerator / numerator.sum()).clamp_min(1e-12)
        p = joint * (4.0 if step < 150 else 1.0)
        loss = (p * (p.log() - q.log())).sum()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            projection.sub_(projection.mean(0))
    return projection.detach().cpu().numpy()


def build_model(model_name, observed_dim, seed):
    torch.manual_seed(seed)
    if model_name == "Gaussian | stationary RBF":
        prior = PairedRBFSpectralPrior(LATENT_DIM)
    elif model_name == "Gaussian | paired non-stationary":
        prior = CorrelatedGaussianJointSpectralPrior(LATENT_DIM)
    elif model_name == f"{NUM_MIXTURE_COMPONENTS}-Gaussian mixture | stationary":
        prior = PairedLocallyPeriodicMixturePrior(
            LATENT_DIM, NUM_MIXTURE_COMPONENTS
        )
    elif model_name == f"{NUM_MIXTURE_COMPONENTS}-Gaussian mixture | paired non-stationary":
        prior = CorrelatedGaussianMixtureJointSpectralPrior(
            LATENT_DIM, NUM_MIXTURE_COMPONENTS
        )
    elif model_name == "Implicit neural | stationary":
        prior = ImplicitNeuralSpectralPrior(
            LATENT_DIM, noise_dim=IMPLICIT_NOISE_DIM,
            hidden_dims=IMPLICIT_HIDDEN_DIMS,
        )
    elif model_name == "Implicit neural | paired non-stationary":
        prior = ImplicitJointSpectralPrior(
            LATENT_DIM, noise_dim=IMPLICIT_NOISE_DIM,
            hidden_dims=IMPLICIT_HIDDEN_DIMS,
        )
    else:
        raise KeyError(f"Unknown model {model_name!r}")

    # Prior construction can consume random numbers. Reset separately so all
    # six GPLVMs start from exactly the same encoder parameters.
    torch.manual_seed(seed + 10_000)
    return PairedRFFGPLVM(
        observed_dim=observed_dim,
        latent_dim=LATENT_DIM,
        num_frequencies=NUM_FOURIER_FREQUENCIES,
        spectral_prior=prior,
    ).to(device)


def run_dataset_experiments(dataset_name, seed):
    data = load_balanced_dataset(dataset_name, seed)
    results = {}
    for model_index, model_name in enumerate(MODEL_NAMES):
        print(f"\n{dataset_name} | {model_name}")
        model = build_model(model_name, data["train_images"].shape[1], seed)
        history = train_model(
            model, data["train_images"], epochs=NUM_EPOCHS,
            learning_rate=2e-3, beta_warmup_epochs=200, print_every=PRINT_EVERY,
        )
        z_train = latent_means(model, data["train_images"])
        z_test = latent_means(model, data["test_images"])
        knn_result = tune_knn(
            z_train, data["train_labels"], z_test, data["test_labels"]
        )
        clustering_result = latent_clustering_metrics(
            z_train, z_test, data["test_labels"]
        )
        results[model_name] = {
            "model": model, "history": history,
            "z_train": z_train, "z_test": z_test,
            "best_knn_k": knn_result["best_k"],
            "knn_cv_accuracy": knn_result["cv_accuracy"],
            "knn_accuracy": knn_result["test_accuracy"],
            "clustering_metrics": clustering_result,
            "spectral_summary": model.spectral_prior.summary(),
        }
        print(
            f"best k={knn_result['best_k']} | "
            f"CV accuracy={knn_result['cv_accuracy']:.4f} | "
            f"test accuracy={knn_result['test_accuracy']:.4f} | "
            f"AMI={clustering_result['AMI mean']:.4f} | "
            f"ARI={clustering_result['ARI mean']:.4f} | "
            f"silhouette={clustering_result['class silhouette']:.4f}"
        )
    return {"name": dataset_name, "data": data, "results": results}


def tsne_for_result(result, data, seed):
    z = np.concatenate((result["z_train"], result["z_test"]), axis=0)
    labels = np.concatenate((data["train_labels"], data["test_labels"]))
    rng = np.random.default_rng(seed)
    selected = rng.choice(len(z), size=min(TSNE_MAX_POINTS, len(z)), replace=False)
    projection = tsne_projection(z[selected], steps=750, seed=seed)
    return projection, labels[selected]


def plot_dataset_results(experiment, seed):
    family_order = ["Gaussian", "Gaussian mixture", "Implicit neural"]
    stationarity_order = ["stationary", "non-stationary"]
    fig, axes = plt.subplots(2, 3, figsize=(17, 10))
    for row, stationarity in enumerate(stationarity_order):
        for column, family in enumerate(family_order):
            ax = axes[row, column]
            model_name = next(
                spec["name"] for spec in MODEL_SPECS
                if spec["family"] == family
                and spec["stationarity"] == stationarity
            )
            result = experiment["results"][model_name]
            clustering = result["clustering_metrics"]
            # The same examples and t-SNE initialization are used in every panel.
            projection, projected_labels = tsne_for_result(
                result, experiment["data"], seed
            )
            scatter = ax.scatter(
                projection[:, 0], projection[:, 1], c=projected_labels,
                cmap="tab10", s=11, alpha=0.75,
            )
            ax.set(
                title=(f"{family} — {stationarity}\n"
                       f"best k={result['best_knn_k']}, "
                       f"test accuracy={result['knn_accuracy']:.3f}\n"
                       f"AMI={clustering['AMI mean']:.3f}, "
                       f"silhouette={clustering['class silhouette']:.3f}"),
                xlabel="t-SNE 1", ylabel="t-SNE 2",
            )
    fig.colorbar(scatter, ax=axes, ticks=range(10), label="class")
    fig.suptitle(
        f"{experiment['name']}: matched t-SNE comparison of "
        f"{LATENT_DIM}D GPLVM posterior means", y=0.995,
    )
    plt.show()


## 5. MNIST experiments

Train all three stationary models and their paired non-stationary counterparts on MNIST, then compare the six latent spaces using matched t-SNE panels and held-out classification.


In [ ]:
mnist_experiment = run_dataset_experiments("MNIST", SEED)
plot_dataset_results(mnist_experiment, SEED + 100)


## 6. Final quantitative comparison

All geometry metrics are computed in the original standardized $K$-dimensional GPLVM space, never in the two-dimensional t-SNE display. Higher test accuracy, AMI, ARI, and silhouette indicate better class organization. AMI corrects mutual information for chance agreement; ARI measures agreement between pairs of cluster assignments; silhouette measures within-class compactness relative to between-class separation. The final ELBO is retained as an optimization diagnostic. The second table reports matched non-stationary-minus-stationary differences within each family.


In [ ]:
comparison_rows = []
for experiment in (mnist_experiment,):
    for model_name, result in experiment["results"].items():
        comparison_rows.append({
            "dataset": experiment["name"],
            "spectral model": model_name,
            "kernel family": MODEL_METADATA[model_name]["family"],
            "stationarity": MODEL_METADATA[model_name]["stationarity"],
            "latent dimension K": LATENT_DIM,
            "frequency samples L": NUM_FOURIER_FREQUENCIES,
            "selected k": result["best_knn_k"],
            "CV accuracy": result["knn_cv_accuracy"],
            "test accuracy": result["knn_accuracy"],
            **result["clustering_metrics"],
            "final negative ELBO/pixel": result["history"].loss[-1],
        })
comparison_table = pd.DataFrame(comparison_rows)
display(
    comparison_table.style.format({
        "CV accuracy": "{:.4f}",
        "test accuracy": "{:.4f}",
        "AMI mean": "{:.4f}",
        "AMI std": "{:.4f}",
        "ARI mean": "{:.4f}",
        "ARI std": "{:.4f}",
        "class silhouette": "{:.4f}",
        "final negative ELBO/pixel": "{:.5f}",
    }).set_caption("Stationary and non-stationary spectral GPLVM comparison")
)

metric_columns = ["test accuracy", "AMI mean", "ARI mean", "class silhouette"]
family_order = ["Gaussian", "Gaussian mixture", "Implicit neural"]
paired_difference_rows = []
for family in family_order:
    family_results = comparison_table.loc[
        comparison_table["kernel family"] == family
    ].set_index("stationarity")
    paired_difference_rows.append({
        "kernel family": family,
        **{
            f"delta {metric}": (
                family_results.loc["non-stationary", metric]
                - family_results.loc["stationary", metric]
            )
            for metric in metric_columns
        },
    })
paired_difference_table = pd.DataFrame(paired_difference_rows).set_index("kernel family")
display(
    paired_difference_table.style.format("{:+.4f}").set_caption(
        "Matched improvement: non-stationary minus stationary"
    )
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
bar_positions = np.arange(len(family_order))
bar_width = 0.36
for ax, metric in zip(axes.flat, metric_columns):
    stationary_values, nonstationary_values = [], []
    for family in family_order:
        family_results = comparison_table.loc[
            comparison_table["kernel family"] == family
        ].set_index("stationarity")
        stationary_values.append(family_results.loc["stationary", metric])
        nonstationary_values.append(family_results.loc["non-stationary", metric])
    ax.bar(
        bar_positions - bar_width / 2, stationary_values,
        width=bar_width, label="stationary", color="tab:blue",
    )
    ax.bar(
        bar_positions + bar_width / 2, nonstationary_values,
        width=bar_width, label="non-stationary", color="tab:orange",
    )
    ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.5)
    ax.set_xticks(bar_positions, family_order, rotation=15, ha="right")
    ax.set_title(metric)
axes[0, 0].legend()
fig.suptitle("MNIST latent-space class separation before t-SNE")
fig.tight_layout()
plt.show()


## 7. Interpretation

- The stationary RBF spectrum imposes a smooth, zero-centered geometry; its paired-Gaussian counterpart can depend separately on two latent locations.
- The stationary mixture introduces explicit spectral modes and weights; the paired mixture additionally learns within-component dependence between $\mathbf w_1$ and $\mathbf w_2$ while marginalizing the component index.
- The implicit pair removes the parametric joint-Gaussian assumption by learning a neural pushforward distribution over $(\mathbf w_1,\mathbf w_2)$.
- Classification labels never enter GPLVM training. The active $k=1$ classifier, AMI, ARI, and silhouette metrics evaluate the geometry only after unsupervised representation learning.
- t-SNE can reveal qualitative clusters but may distort distances. All quantitative separation metrics are therefore computed in the original standardized $K$-dimensional latent space.
- AMI and ARI depend on the ten-cluster $k$-means partition and are averaged over several initializations; silhouette uses the known classes directly and does not require clustering.
- Because GPLVM latent coordinates are learned jointly with the kernel, the encoder can absorb some non-stationarity through a latent-space warp. The paired models are therefore more expressive, but are not guaranteed to improve classification.
- Results depend on stochastic optimization and finite RFF sampling. For a formal comparison, repeat the experiment across multiple seeds and report uncertainty.
